# Comparaison des dynamiques Swendsen-Wang (Hybride CuPy / SciPy)

Ce notebook implémente l'approche hybride optimisée (Maths GPU CuPy + Graphes CPU SciPy) pour vaincre les goulots d'étranglement de l'A100 sur les graphes peu denses.

In [ ]:
!pip install cupy-cuda12x # A ajuster selon l'environnement si ce n'est pas sur Colab
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm.notebook import tqdm

# Paramètres
d = 2
p = 0.65
n = 10000
L = int(np.round(n ** (1/d)))
T = 1000
N_samples = 10

In [ ]:
def generate_graph(L):
    N = L * L
    edges = []
    for y in range(L):
        for x in range(L):
            edges.append([y*L + x, y*L + (x+1)%L])
    for y in range(L):
        for x in range(L):
            edges.append([y*L + x, ((y+1)%L)*L + x])
    for y in range(L):
        for x in range(L):
            edges.append([y*L + x, ((y+1)%L)*L + (x+1)%L])
            
    edges = cp.array(edges)
    
    white_triangles = []
    black_triangles = []
    for y in range(L):
        for x in range(L):
            idx = y*L + x
            h_edge = idx
            v_edge = N + y*L + (x+1)%L
            d_edge = 2*N + idx
            white_triangles.append([h_edge, v_edge, d_edge])
            
            v_edge_b = N + idx
            h_edge_b = ((y+1)%L)*L + x
            d_edge_b = 2*N + idx
            black_triangles.append([v_edge_b, h_edge_b, d_edge_b])
            
    return edges, cp.array(white_triangles), cp.array(black_triangles)

In [ ]:
def SW_step(sigma, W, edges, edges_cpu, mode, up, triangles_A, triangles_B=None):
    frozen = cp.zeros(len(edges), dtype=bool)
    satisfied = (W * sigma[edges[:,0]] * sigma[edges[:,1]]) > 0
    
    if mode == 'edges':
        freeze_prob = 1 - cp.exp(-up)
        frozen = satisfied & (cp.random.rand(len(edges)) < freeze_prob)
    else:
        if mode == 'white':
            triangles = triangles_A
            au = 1 - cp.exp(-2 * up)
        else:
            triangles = cp.vstack((triangles_A, triangles_B))
            au = 1 - cp.exp(-up)
            
        W_tri = W[triangles]
        type_A = cp.prod(W_tri, axis=1) > 0
        sat_tri = satisfied[triangles]
        sat_count = cp.sum(sat_tri, axis=1)
        
        rand_vals = cp.random.rand(len(triangles))
        freeze_A = type_A & (sat_count == 3) & (rand_vals < au)
        freeze_B = (~type_A) & (sat_count == 2) & (rand_vals < au)
        
        # Freeze A en une opération vectorisée
        frozen[triangles[freeze_A].flatten()] = True
            
        # Freeze B: L'astuce ici est d'éviter options.sort(axis=1) qui est lent sur GPU
        B_idx = cp.where(freeze_B)[0]
        if len(B_idx) > 0:
            sat_B = sat_tri[B_idx]
            row_idx, col_idx = cp.where(sat_B)
            pick = cp.random.randint(0, 2, size=len(B_idx))
            flat_idx = cp.arange(len(B_idx)) * 2 + pick
            chosen_col = col_idx[flat_idx]
            chosen_edges = triangles[B_idx, chosen_col]
            frozen[chosen_edges] = True

    # HYBRIDATION CPU : SciPy est 100x plus rapide que CuPy pour les composantes connexes sur petit graphe
    frozen_cpu = frozen.get()
    row_cpu = edges_cpu[frozen_cpu, 0]
    col_cpu = edges_cpu[frozen_cpu, 1]
    data_cpu = np.ones(len(row_cpu), dtype=np.float32)
    
    N_nodes = len(sigma)
    adj_cpu = csr_matrix((data_cpu, (row_cpu, col_cpu)), shape=(N_nodes, N_nodes))
    n_comp, labels_cpu = connected_components(adj_cpu, directed=False)
    
    lcc_frac = float(np.max(np.bincount(labels_cpu))) / N_nodes
    
    # Retour sur GPU pour le masquage final (flip aléatoire sans cp.choice qui est lent)
    labels = cp.array(labels_cpu)
    flip = cp.random.randint(0, 2, size=n_comp) * 2 - 1
    sigma = sigma * flip[labels]
    
    return sigma, lcc_frac

In [ ]:
edges_gpu, white_tri_gpu, black_tri_gpu = generate_graph(L)
edges_cpu = edges_gpu.get() # Cache CPU pour l'hybridation
up = cp.log(p / (1 - p))
modes = ['edges', 'white', 'half-half']

overlap_history = {m: np.zeros((N_samples, T)) for m in modes}
lcc_history = {m: np.zeros((N_samples, T)) for m in modes}

for sample in tqdm(range(N_samples), desc="Samples"):
    Sigma = cp.random.randint(0, 2, size=L*L) * 2 - 1
    
    same_comm = Sigma[edges_gpu[:,0]] == Sigma[edges_gpu[:,1]]
    correct_obs = cp.random.rand(len(edges_gpu)) < p
    W = cp.where(same_comm == correct_obs, up, -up)
    
    for mode in modes:
        sigma = cp.random.randint(0, 2, size=L*L) * 2 - 1
        
        for t in range(T):
            sigma, lcc = SW_step(sigma, W, edges_gpu, edges_cpu, mode, up, white_tri_gpu, black_tri_gpu)
            ov = float(cp.abs(cp.mean(sigma * Sigma)))
            
            overlap_history[mode][sample, t] = ov
            lcc_history[mode][sample, t] = lcc

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

colors = {'edges': 'blue', 'white': 'orange', 'half-half': 'green'}
labels = {'edges': 'SW-Edges', 'white': 'SW-Triangles (Blancs)', 'half-half': 'SW-Triangles (Half-Half)'}

for mode in modes:
    mean_ov = np.mean(overlap_history[mode], axis=0)
    std_ov = np.std(overlap_history[mode], axis=0)
    ax1.plot(mean_ov, label=labels[mode], color=colors[mode])
    ax1.fill_between(range(T), mean_ov - std_ov, mean_ov + std_ov, color=colors[mode], alpha=0.2)
    
    mean_lcc = np.mean(lcc_history[mode], axis=0)
    std_lcc = np.std(lcc_history[mode], axis=0)
    ax2.plot(mean_lcc, label=labels[mode], color=colors[mode])
    ax2.fill_between(range(T), mean_lcc - std_lcc, mean_lcc + std_lcc, color=colors[mode], alpha=0.2)

ax1.set_title('Overlap (Recouvrement) vs Itérations')
ax1.set_xlabel('Itération')
ax1.set_ylabel('Overlap moyen')
ax1.legend()
ax1.grid(True)

ax2.set_title('Taille de la plus grande composante (LCC) vs Itérations')
ax2.set_xlabel('Itération')
ax2.set_ylabel('Proportion de points dans la LCC')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()